# RAC Vision: detector de chalecos reflectantes

Entrena YOLOv8n para encontrar **personas** y **chalecos** en la imagen. La app marca
"sin chaleco" a una persona cuyo torso no tiene un chaleco encima.

Datos: [Construction-PPE de Ultralytics](https://docs.ultralytics.com/datasets/detect/construction-ppe)
(1416 fotos de obra, AGPL-3.0). Se usan solo las clases `Person` y `vest`.

Antes de empezar: *Entorno de ejecución → Cambiar tipo → GPU T4*. Después, *Ejecutar todas*.
Al final se descarga `detector_chaleco.onnx`.

## 1. Librerías y GPU

In [ ]:
!nvidia-smi -L
!pip install -q "ultralytics>=8.3,<8.4"

## 2. Bajar el dataset y quedarse con persona y chaleco

In [ ]:
import shutil, zipfile, urllib.request
from collections import Counter
from pathlib import Path

URL = "https://github.com/ultralytics/assets/releases/download/v0.0.0/construction-ppe.zip"
urllib.request.urlretrieve(URL, "/content/construction-ppe.zip")
zipfile.ZipFile("/content/construction-ppe.zip").extractall("/content/ppe")
SRC = next(p.parent for p in Path("/content/ppe").rglob("images") if (p / "train").exists())
print("Dataset en", SRC)

# Clases del dataset: 2 = vest, 6 = Person (construction-ppe.yaml de Ultralytics)
MAPA = {6: 0, 2: 1}          # 0 = persona, 1 = chaleco
DST = Path("/content/chaleco_yolo")
conteo = Counter()
for split in ("train", "val", "test"):
    (DST / "images" / split).mkdir(parents=True, exist_ok=True)
    (DST / "labels" / split).mkdir(parents=True, exist_ok=True)
    for img in (SRC / "images" / split).glob("*"):
        lbl = SRC / "labels" / split / (img.stem + ".txt")
        lineas = []
        for l in (lbl.read_text().splitlines() if lbl.exists() else []):
            partes = l.split()
            if partes and int(partes[0]) in MAPA:
                c = MAPA[int(partes[0])]
                lineas.append(" ".join([str(c)] + partes[1:]))
                conteo[c] += 1
        shutil.copy(img, DST / "images" / split / img.name)
        (DST / "labels" / split / (img.stem + ".txt")).write_text("\n".join(lineas))
(DST / "data.yaml").write_text(f"path: {DST}\ntrain: images/train\nval: images/val\ntest: images/test\nnames:\n  0: persona\n  1: chaleco\n")
print("Cajas: personas", conteo[0], "| chalecos", conteo[1])

## 3. Entrenar

In [ ]:
from ultralytics import YOLO
modelo = YOLO("yolov8n.pt")
modelo.train(data=str(DST / "data.yaml"), epochs=60, imgsz=640, batch=32, patience=15,
             project="/content/runs", name="chaleco", exist_ok=True)
YOLO("/content/runs/chaleco/weights/best.pt").val(data=str(DST / "data.yaml"), split="test")

## 4. Exportar y descargar

In [ ]:
from google.colab import files
final = YOLO("/content/runs/chaleco/weights/best.pt")
onnx = final.export(format="onnx", imgsz=640, opset=12, simplify=True, dynamic=False)
shutil.copy(onnx, "/content/detector_chaleco.onnx")
files.download("/content/detector_chaleco.onnx")